In [9]:
# src/sh_ssw_methods/polarT.py

In [10]:
import numpy as np
import pandas as pd
import glob

import xarray as xr

from pathlib import Path

In [99]:
from __future__ import annotations

from pathlib import Path
from utils.event_iden_funcs import build_events_df

In [135]:
from preprocessing.prepare_polarT_files import output_polarT_prefiles

In [102]:
from preprocessing.return_fsw_dates import detect_fsw_SH

- ERA5 temperature file for a given level (for myself)

In [17]:
# import module path
import sys
sys.path.insert(1, '/net/cfc/s2s/rachwu/scripts/tools/read_hc/')
sys.path.insert(2, '/net/cfc/s2s/rachwu/scripts/tools/read_data/')
sys.path.insert(3, '/net/cfc/s2s/rachwu/scripts/WP2/modules')
sys.path.insert(4, '/net/cfc/s2s/rachwu/scripts/tools/wind_events/')

In [18]:
import mod_read_erai as mera

In [19]:
def return_u300(start_date, end_date, ilat, ilon, ilev, varname):
    time_gph, lon, lat, lev, u_era5 = mera.return_var_era5(varname, start_date, end_date, ilat, ilon, ilev)
    u_era5 = u_era5.groupby(u_era5.time.dt.floor('1D')).mean().rename({'floor':'time'})
    time_era5 = u_era5.time
    
    return time_era5, lat, lon, u_era5

In [31]:
def build_multiyear_T(
    start_year=1979,
    end_year=2023,
    ilat=slice(0,-90), 
    ilon=slice(0,360),
    ilev=1000,
    varname="temp",
    out_dir="./processed",
    overwrite=False,
):
    """
    Read ERA5 wind data year by year, compute daily means, and merge into one file.
    Saves as: era5_u_<level>hPa_<start>_<end>.nc
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    outfile = out_dir / f"era5_u_{int(ilev)}hPa_{start_year}_{end_year}.nc"

    if outfile.exists() and not overwrite:
        print(f" Found existing file: {outfile}")
        return xr.open_dataarray(outfile)

    all_years = []

    for yr in range(start_year, end_year + 1):
        print(f"Processing ERA5 {varname} for {yr}...")
        try:
            time_era5, lat, lon, u_yr = return_u300(f"{yr}-01-01", f"{yr}-12-31", ilat=ilat, ilon=ilon, ilev=ilev, varname=varname)
            all_years.append(u_yr)
        except Exception as e:
            print(f"Skipping {yr}: {e}")

    # Combine along time dimension
    u_all = xr.concat(all_years, dim="time").sortby("time")

    # Save
    u_all.to_netcdf(outfile)
    print(f" Saved merged file: {outfile}")

    return u_all


In [32]:
T_all = build_multiyear_T(
    start_year=1979,
    end_year=2023,
    ilev=1000,
    out_dir="./processed",
    overwrite=False,
)

Processing ERA5 temp for 1979...
Processing ERA5 temp for 1980...
Processing ERA5 temp for 1981...
Processing ERA5 temp for 1982...
Processing ERA5 temp for 1983...
Processing ERA5 temp for 1984...
Processing ERA5 temp for 1985...
Processing ERA5 temp for 1986...
Processing ERA5 temp for 1987...
Processing ERA5 temp for 1988...
Processing ERA5 temp for 1989...
Processing ERA5 temp for 1990...
Processing ERA5 temp for 1991...
Processing ERA5 temp for 1992...
Processing ERA5 temp for 1993...
Processing ERA5 temp for 1994...
Processing ERA5 temp for 1995...
Processing ERA5 temp for 1996...
Processing ERA5 temp for 1997...
Processing ERA5 temp for 1998...
Processing ERA5 temp for 1999...
Processing ERA5 temp for 2000...
Processing ERA5 temp for 2001...
Processing ERA5 temp for 2002...
Processing ERA5 temp for 2003...
Processing ERA5 temp for 2004...
Processing ERA5 temp for 2005...
Processing ERA5 temp for 2006...
Processing ERA5 temp for 2007...
Processing ERA5 temp for 2008...
Processing

In [33]:
temp_sh_era5 = xr.open_dataset('processed/era5_temp_sh_10hPa_1979_2023.nc')

In [35]:
temp_sh_era5 = temp_sh_era5['var130']

In [36]:
temp_sh_era5

<xarray.DataArray 'var130' (time: 16436, lat: 46, lon: 180)> Size: 544MB
[136090080 values with dtype=float32]
Coordinates:
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
  * lat      (lat) float64 368B 0.0 -2.0 -4.0 -6.0 ... -84.0 -86.0 -88.0 -90.0
    plev     float64 8B ...
  * time     (time) datetime64[ns] 131kB 1979-01-01 1979-01-02 ... 2023-12-31
Attributes:
    table:    128

- preprocess data

Files needed:
- Tmid_polar - T8090, T6070, temperature averaged over latitude bands
- polarT anom - polar cap temperature anomalies
- fw_dates - final warming dates

In [92]:
ds_bands, polarT, polarT_anom =  output_polarT_prefiles(temp_sh_era5)

 Saved to processed/Tmid_polar_10hPa_daily_1979_2023_era5.nc
 Saved to processed/polarT_10hPa_daily_1979_2023_era5.nc
Saved to processed/polarT_anom_10hPa_daily_1979_2023_era5.nc


In [80]:
fu1060s = '/net/cfc/s2s/rachwu/scripts/sh_ssw_impact/aparc_project/SouthernWarmings_module/data/u1060s_era5_1959_2023.nc'

In [82]:
u1060s = xr.open_dataset(fu1060s)
u1060s_daily = u1060s.resample(time='1D').mean()

In [103]:
fsw = detect_fsw_SH(
    da=u1060s_daily,          # zonal wind (time)
    max_westerly_return=10,
    smooth=5,
    level_hpa=10,
    save_file=True,
    out_dir="./processed"
)

 Saved to processed/FSW_dates_SH_10hPa_60S_1959_2023_era5.nc


- prepare xarray

In [3]:
# T6080, T8090
def aug_nov_by_year(ds):
    """
    ds: xarray.Dataset with daily 'time' and variables 'T8090S', 'T6070S'
    returns: Dataset with dims ('year','day') and a 'date' variable for reference.
    """
    ds = ds.sortby('time')

    # Keep Aug–Nov only
    sub = ds.sel(time=ds.time.dt.month.isin([8, 9, 10, 11]))

    # Index within each year: 0..121 (Aug–Nov has 122 days in non-leap years)
    ti = sub.indexes['time']  # pandas.DatetimeIndex
    day_idx = pd.Series(np.arange(ti.size), index=ti).groupby(ti.year).cumcount().to_numpy()

    # Add coords for pivoting
    sub = sub.assign_coords(
        year=('time', sub.time.dt.year.values),
        day=('time', day_idx),
        date=('time', sub.time.values)  # keep the actual datestamp as a variable
    )

    # Pivot time -> (year, day)
    wide = sub[['T8090S', 'T6070S', 'date']].set_index(time=['year', 'day']).unstack('time')

    # Tidy output
    out = xr.Dataset(
        {
            'T8090S': wide['T8090S'],
            'T6070S': wide['T6070S'],
            'date':   wide['date'],  # same shape (year, day), useful for debugging / alignment
        }
    )
    # Optional: name dims explicitly (xarray may already name them 'year' and 'day')
    out = out.transpose('year', 'day')

    # (Optional) Sanity check: every year should have 122 days
    # If you have missing days, this will vary; handle reindexing per-year if needed.
    # counts = out['date'].count(dim='day').to_pandas()

    return out

In [4]:
def output_Tvars(fpath, start_date="1979-01-01", end_date="2021-12-31"):
    xT = xr.open_dataset(fpath)

    # select time period
    xT = xT.sel(time=slice(start_date, end_date))
    
    # extract only aug to nov
    T_augnov = aug_nov_by_year(xT)
    
    # compute climatologies
    cT8090 = T_augnov['T8090S'].isel(year=slice(0, 43)).mean('year')
    cT6070 = T_augnov['T6070S'].isel(year=slice(0, 43)).mean('year')
    
    # delta climatology (80–90S minus 60–70S)
    cdelT = cT8090 - cT6070
    
    # Anomalies (broadcast day climatology across years)
    T8090anom = T_augnov['T8090S'] - cT8090
    T6070anom = T_augnov['T6070S'] - cT6070

    # Indices where climatological delta ≤ 0
    cidx = np.where(cdelT.values <= 0)[0]
    ncidx = cidx.size
    
    if ncidx > 0:
        first_idx, last_idx = cidx[0], cidx[-1]
        # print(first_idx, last_idx)
    
    # Year-by-year daily difference
    delT = T_augnov['T8090S'] - T_augnov['T6070S']          # (year, day)
    
    # Std dev across years for each day
    stddv = delT.std(dim='year')                          # (day,)
    
    return T_augnov, cdelT, delT, stddv

In [5]:
# final warming dates
def output_fwidx(fwdate_path, T_augnov):
    fwdate = xr.load_dataarray(fwdate_path)
    
    fwmask = T_augnov['date'] == fwdate.rename({'season_year':'year'}).broadcast_like(T_augnov['date'])
    fw_idx = fwmask.argmax(dim='day') 

    return fw_idx

In [106]:
# polarT
def return_polarT(polarT_path, start_date, end_date):
    da = xr.load_dataarray(polarT_path)

    # Assume da is your DataArray (name 'T6090S'), with a daily 'time' coord
    da_aug_nov = da.sel(time=da.time.dt.month.isin([8, 9, 10, 11]))
    da_aug_nov = da_aug_nov.sel(time=slice(start_date, end_date)) 

    ti = da_aug_nov.indexes['time']  # DatetimeIndex
    day_idx = pd.Series(range(ti.size), index=ti).groupby(ti.year).cumcount().to_numpy()
    
    polarT = da_aug_nov.assign_coords(
        year=('time', da_aug_nov.time.dt.year.values),
        day =('time', day_idx),
    ).set_index(time=['year','day']).unstack('time')  # dims now ('year','day')

    return polarT

In [109]:
istart_date = '1979-01-01'
iend_date = '2023-12-31'

years = np.arange(1979, 2023+1,1)

In [121]:
Tband_file = "processed/Tmid_polar_10hPa_daily_1979_2023_era5.nc"
polarT_file = "processed/polarT_anom_10hPa_daily_1979_2023_era5.nc"
fsw_file = "processed/FSW_dates_SH_10hPa_60S_1959_2023_era5.nc"

In [122]:
def prep_input_polarT(Tband_file, polarT_file, fsw_file, istart_date, iend_date):
    """
    Prepare all input variables required for the SSW detection algorithms based on
    polar-cap temperature gradients (e.g., for detect_ssw_tgrad).

    This function loads preprocessed datasets from disk — including the temperature
    band data (for computing ΔT and climatology), the polar-cap temperature, and the
    final stratospheric warming (FSW) date indices — and returns them as ready-to-use
    xarray and numpy objects within a specified time range.

    Parameters
    ----------
    Tband_file : str or Path
        Path to the NetCDF file containing temperature-band data (e.g., T80–90S and T60–70S).
        Used by `output_Tvars()` to compute ΔT, climatological mean, and standard deviation.
    polarT_file : str or Path
        Path to the NetCDF file containing polar-cap (60–90S) mean temperature.
    fsw_file : str or Path
        Path to the NetCDF or .npy file containing final stratospheric warming (FSW) dates.
        Used by `output_fwidx()` to produce per-year index positions for each event.
    istart_date : str
        Start date (inclusive) of the analysis period (e.g., '1979-08-01').
    iend_date : str
        End date (inclusive) of the analysis period (e.g., '2023-11-30').

    Returns
    -------
    tuple
        (T_augnov, cdelT, delT, stddv, polarT, fw_idx)
        where:
            T_augnov : xr.Dataset
                Temperature dataset for the selected time window.
            cdelT : xr.DataArray
                Day-of-year climatological mean of ΔT.
            delT : xr.DataArray
                Daily temperature difference between 80–90S and 60–70S.
            stddv : xr.DataArray
                Day-of-year standard deviation of ΔT.
            polarT : xr.DataArray
                Daily polar-cap (60–90S) mean temperature.
            fw_idx : np.ndarray
                Array of FSW index positions per year for event alignment.

    Notes
    -----
    This function is typically called before SSW detection algorithms such as
    `detect_ssw_tgrad()` or `polarT1_lim()` to assemble all necessary temperature
    and reference data in memory.
    """

    T_augnov, cdelT, delT, stddv = output_Tvars(Tband_file, start_date=istart_date, end_date=iend_date)
    polarT = return_polarT(polarT_file, istart_date, iend_date)
    fw_idx = output_fwidx(fsw_file, T_augnov)

    return T_augnov, cdelT, delT, stddv, polarT, fw_idx

In [123]:
T_augnov, cdelT, delT, stddv, polarT, fw_idx = prep_input_polarT(Tband_file, polarT_file, fsw_file, istart_date, iend_date)

- detection algorithm

In [113]:
# iden algorithm
def polarT1_lim(T_augnov, fw_idx, delT, cdelT, stddv, polarT):
    
    xtime1 = T_augnov['date']
    
    count = np.full((len(years), 122), -999, dtype=int)
    sel_dates = []
    
    for i, yr in enumerate(years):
        if fw_idx[i] < 0:
            continue
    
        # Loop j = 0 .. eidx-20  (avoid last 20 days before FW)
        jmax = int(fw_idx[i]) - 20
        jmax = min(jmax, delT.shape[1] - 5)   # prevent partial windows
        if jmax < 0:
            continue
    
        for j in range(0, jmax + 1):
            # 1) sign reversal (positive) & 5-day persistence
            if delT[i, j].item() > 0:
                win = delT[i, j:j+5].values  # length 5
                if win.shape[0] != 5:
                    continue                  # safety, but jmax should prevent this
    
                if np.all(win >= 0):
                    # print('cond1: %s' % (win))
                    maxidx = int(np.argmax(win))   # index in the 5-day window
                    midx   = j + maxidx            # absolute time index within season
    
                    # 2) anomaly vs climatology stddev
                    anom = (delT[i, midx] - cdelT[midx]).item()
                    
                    if anom >= float(stddv[midx]):
                        # print(xtime1[i,j].values)
                        # print('cond2: anom=%s, std=%s' % (anom, stddv[midx].values))
                        # print('polarT=%s' % polarT[i, j].item())
                        # 3) polar cap temperature positive at the window start j
                        if polarT[i, j].item() > 0:
                            
                            # print('cond3: polarT=%s' % polarT[i, j].item())
                            count[i, j] = 1
                            sel_dates.append(xtime1[i,j].values)

    # make sure events are 60 days apart
    starts = np.r_[True, np.diff(sel_dates) >= np.timedelta64(60, 'D')]
    event_dates = np.array(sel_dates)[starts]
    
    return event_dates
                        

In [114]:
def polarT2_lim(T_augnov, fw_idx, delT, cdelT, stddv, polarT, pers=5):
    
    xtime1 = T_augnov['date']
    
    count = np.full((len(years), 122), -999, dtype=int)
    sel_dates = []
    
    for i, yr in enumerate(years):
        if fw_idx[i] < 0:
            continue
    
        # Loop j = 0 .. eidx-20  (avoid last 20 days before FW)
        jmax = int(fw_idx[i]) - 20
        jmax = min(jmax, delT.shape[1] - 5)   # prevent partial windows
        if jmax < 0:
            continue
    
        for j in range(0, jmax + 1):
            # 1) sign reversal (positive) & 5-day persistence
            if delT[i, j].item() > 0:
                w_delT = delT[i, j:j+pers].values  # length 5
                if w_delT.shape[0] != 5:
                    continue                  # safety, but jmax should prevent this
    
                if np.all(w_delT >= 0):
                    print('cond1: %s' % (w_delT))

                    # 2) anomaly ≥ 1σ for all 5 days
                    w_c   = cdelT[j:j+pers].values
                    w_std = stddv[j:j+pers].values
                    
                    anom = (w_delT - w_c) / w_std
                    
                    if not np.all(anom >= 1):
                        continue

                    print(xtime1[i,j].values)
                    print('cond2: anom=%s, std=%s' % (anom, w_std))
                    print('polarT=%s' % polarT[i, j].item())
                    
                    # 3) polar cap temperature positive at the window start j
                    w_polar = polarT[i, j + pers].values
                    if not (np.all(np.isfinite(w_polar)) and np.all(w_polar >= 0)):
                        continue    
                    print('cond3: polarT=%s' % polarT[i, j].item())
                    count[i, j] = 1
                    sel_dates.append(xtime1[i,j].values)

    # make sure events are 60 days apart
    starts = np.r_[True, np.diff(sel_dates) >= np.timedelta64(60, 'D')]
    event_dates = np.array(sel_dates)[starts]
    
    return event_dates


In [132]:
def detect_ssw_tgrad(
    T_augnov: xr.Dataset,
    *,
    algo: str = "T1",               # choose between "T1" or "T2"
    fw_idx: np.ndarray | None = None,
    delT: xr.DataArray | None = None,
    cdelT: xr.DataArray | None = None,
    stddv: xr.DataArray | None = None,
    polarT: xr.DataArray | None = None,
    persist_days: int = 5,
    min_gap_days: int = 60,
    thres_sigma: float = 1.0,
    level_hpa: float | None = None,
    latitude: str | None = "-80_to_-90_-60_to_-70",
    data_source: str | None = None,
) -> tuple[pd.DataFrame, pd.DatetimeIndex]:
    """
    Wrapper for Shen et al. (2022) temperature-gradient–based SSW detection.

    Calls either `polarT1_lim` or `polarT2_lim` depending on `algo`.
    Returns standardized output (events_df, event_dates) compatible with `detect_ssw_u_anom`.

    Parameters
    ----------
    T_augnov : xr.Dataset
        Input dataset containing date/time info and diagnostic arrays.
    algo : str, optional
        "T1" or "T2" algorithm choice.
    fw_idx, delT, cdelT, stddv, polarT : arrays
        Precomputed diagnostics from ERA5 or similar dataset.
    persist_days : int, optional
        Persistence requirement (used by T2).
    min_gap_days : int, optional
        Minimum separation between detected events.
    thres_sigma : float, optional
        Standard deviation threshold (used by T2).
    level_hpa : float, optional
        Pressure level label.
    latitude : str, optional
        Latitude range label.
    data_source : str, optional
        Metadata label for data source.

    Returns
    -------
    events_df : pd.DataFrame
        Standardized event metadata table.
    event_dates : pd.DatetimeIndex
        Detected event onset dates.
    """

    # --- Dispatch to chosen algorithm ---
    if algo.upper() == "T1":
        event_dates = polarT1_lim(T_augnov, fw_idx, delT, cdelT, stddv, polarT)
        method_name = "tgrad_T1"
    elif algo.upper() == "T2":
        event_dates = polarT2_lim(T_augnov, fw_idx, delT, cdelT, stddv, polarT, pers=persist_days)
        method_name = "tgrad_T2"
    else:
        raise ValueError("Invalid algo: choose 'T1' or 'T2'")

    # Ensure DatetimeIndex output
    event_dates = pd.to_datetime(event_dates)

    # --- Build standardized event dataframe ---
    events_df = build_events_df(
        dates=event_dates,
        method=method_name,
        definition=f"{method_name}_{thres_sigma}σ_{persist_days}d_persist",
        data_source=data_source or "",
        threshold=f"{thres_sigma}σ",
        level_hpa=str(level_hpa),
        latitude=str(latitude),
        notes=f"{method_name}: ≥{persist_days}d persistence; min_gap={min_gap_days}d",
    )

    return events_df, event_dates


In [136]:
events_df, event_dates = detect_ssw_tgrad(
    T_augnov=T_augnov,
    algo="T1",
    fw_idx=fw_idx,
    delT=delT,
    cdelT=cdelT,
    stddv=stddv,
    polarT=polarT,
    level_hpa=10,
    data_source="era5",
)

In [137]:
events_df

,date,method,definition,data_source,threshold,level_hpa,latitude,lat_band,notes
0,1979-10-01,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
1,1982-10-07,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
2,1984-10-05,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
3,1988-09-26,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
4,1989-10-20,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
5,1991-10-14,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
6,1992-09-30,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
7,1993-10-20,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
8,2000-10-05,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
9,2002-09-21,tgrad_T1,tgrad_T1_1.0σ_5d_persist,era5,1.0σ,10,-80_to_-90_-60_to_-70,<NA>,tgrad_T1: ≥5d persistence; min_gap=60d
